In [1]:
import amy
import random
import time
import threading
import ipywidgets as W
from IPython.display import display

### Setup amy

In [2]:
amy.live(default_synths=1, audio_in=0)
amy.reset()

### Use Oscillator

In [3]:
freq = W.FloatSlider(value=220, min=50, max=1000, step=1, description="Freq")

btn_on = W.Button(description="On", button_style="success")
btn_off = W.Button(description="Off", button_style="danger")
playing = {"on": False}

def play(_):
    f = freq.value
    amy.send(osc=0, wave=amy.SAW_DOWN, freq=f, vel=1)
    amy.send(osc=1, wave=amy.SAW_DOWN, freq=f*2, vel=1)

def stop(_):
    amy.send(osc=0, vel=0)
    amy.send(osc=1, vel=0)

def update(_):
    if playing["on"]:
        f = freq.value
        amy.send(osc=0, freq=f)
        amy.send(osc=1, freq=f*2)

btn_on.on_click(play)
btn_off.on_click(stop)
freq.observe(update, names="value")

display(W.VBox([freq, W.HBox([btn_on, btn_off])]))

### Dynamic sequence

In [13]:
import amy, random, threading, time, ipywidgets as W
from IPython.display import display

amy.live(default_synths=0, audio_in=0, startup_bleep=0)
amy.reset()

running = {"on": False}
big_fx = {"on": False}

BASE   = W.IntSlider(value=45, min=36, max=60, step=1, description="Base")
STEP   = W.FloatSlider(value=0.22, min=0.12, max=0.45, step=0.01, description="Step")
CHANGE = W.FloatSlider(value=0.20, min=0.0, max=1.0, step=0.05, description="Change")

midi_to_freq = lambda n: 440.0 * 2 ** ((n - 69) / 12)

def all_off():
    for osc in range(16):
        amy.send(osc=osc, vel=0)

def setup_fx():
    if big_fx["on"]:
        amy.chorus(level=0.55, freq=0.18, amp=0.9)
        amy.reverb(level=0.34, liveness=0.92, damping=0.36, xover_hz=2400)
        amy.echo(level=0.18, delay_ms=360, feedback=0.26, filter_coef=0.10)
    else:
        amy.chorus(level=0.36, freq=0.13, amp=0.65)
        amy.reverb(level=0.18, liveness=0.88, damping=0.44, xover_hz=2200)
        amy.echo(level=0.08, delay_ms=300, feedback=0.16, filter_coef=0.10)

def play_note(midi, accent=1.0, cutoff=950):
    f = midi_to_freq(midi)
    amy.send(
        osc=0, wave=amy.SAW_DOWN, freq=f*0.998, vel=0.56*accent,
        bp0="0,0,18,1,180,0.78,900,0",
        eg0_type=amy.ENVELOPE_TRUE_EXPONENTIAL,
        filter_type=amy.FILTER_LPF, filter_freq=cutoff, resonance=1.0, pan=0.35
    )
    amy.send(
        osc=1, wave=amy.SAW_DOWN, freq=f*1.002, vel=0.56*accent,
        bp0="0,0,22,1,220,0.72,1000,0",
        eg0_type=amy.ENVELOPE_TRUE_EXPONENTIAL,
        filter_type=amy.FILTER_LPF, filter_freq=cutoff*1.05, resonance=1.0, pan=0.65
    )
    amy.send(
        osc=2, wave=amy.PULSE, freq=f/2, vel=0.24*accent,
        bp0="0,0,30,1,260,0",
        eg0_type=amy.ENVELOPE_TRUE_EXPONENTIAL,
        filter_type=amy.FILTER_LPF, filter_freq=cutoff*0.7, resonance=0.8, pan=0.5
    )

def maybe_variant(pattern, p):
    out = pattern[:]
    if random.random() < p:
        i = random.randrange(len(out))
        out[i] = random.choice([0, 3, 5, 7, 10, 12])
    if random.random() < p * 0.6:
        out[-1] += 12
    return out

def loop():
    base_patterns = [
        [0, 3, 7, 10, 12, 10, 7, 3],
        [0, 3, 7, 12, 10, 7, 3, 0],
        [0, 5, 7, 10, 12, 10, 7, 5],
    ]
    accents = [1.15, 0.9, 1.0, 0.92, 1.08, 0.9, 1.0, 0.95]
    pattern = base_patterns[0]
    bar = 0

    while running["on"]:
        setup_fx()

        if bar % 4 == 0:
            base = random.choice(base_patterns)
            pattern = maybe_variant(base, CHANGE.value)

        for i, interval in enumerate(pattern):
            if not running["on"]:
                break

            note = BASE.value + interval
            cutoff = 820 + i * 50 + (20 if big_fx["on"] else 0)
            play_note(note, accent=accents[i], cutoff=cutoff)

            if big_fx["on"] and random.random() < CHANGE.value * 0.35 and i in (3, 7):
                play_note(note + 12, accent=0.55, cutoff=cutoff * 1.1)

            time.sleep(STEP.value)

        bar += 1

def start(_):
    if running["on"]:
        return
    amy.reset()
    setup_fx()
    running["on"] = True
    threading.Thread(target=loop, daemon=True).start()

def stop(_):
    running["on"] = False
    all_off()

def toggle_fx(_):
    big_fx["on"] = not big_fx["on"]
    fx_btn.description = "Big FX On" if big_fx["on"] else "Big FX Off"
    fx_btn.button_style = "warning" if big_fx["on"] else ""

start_btn = W.Button(description="Start", button_style="success")
stop_btn  = W.Button(description="Stop", button_style="danger")
fx_btn    = W.Button(description="Big FX Off")

start_btn.on_click(start)
stop_btn.on_click(stop)
fx_btn.on_click(toggle_fx)

display(W.VBox([
    BASE, STEP, CHANGE,
    W.HBox([start_btn, stop_btn, fx_btn])
]))

### Filter

In [14]:
cutoff = W.FloatSlider(value=1200, min=50, max=12000, step=50, description="Cutoff")
res = W.FloatSlider(value=0.8, min=0.5, max=16.0, step=0.1, description="Res")

btn_on = W.Button(description="Filter On", button_style="success")
btn_off = W.Button(description="Filter Off", button_style="danger")

NUM_OSC = 64
filter_on = {"state": True}


def apply_filter(_=None):
    if not filter_on["state"]:
        return
    for osc in range(NUM_OSC):
        amy.send(
            osc=osc,
            filter_type=amy.FILTER_LPF,
            filter_freq=cutoff.value,
            resonance=res.value,
        )


def enable_filter(_):
    filter_on["state"] = True
    apply_filter()


def disable_filter(_):
    filter_on["state"] = False
    for osc in range(NUM_OSC):
        amy.send(osc=osc, filter_type=0)  # bypass


cutoff.observe(apply_filter, names="value")
res.observe(apply_filter, names="value")

btn_on.on_click(enable_filter)
btn_off.on_click(disable_filter)

display(W.VBox([
    cutoff,
    res,
    W.HBox([btn_on, btn_off])
]))